# GeoLink: tìm siêu tham số

## 1. Chuẩn bị môi trường


In [ ]:
!git clone https://github.com/tranminhduc9/well-log-imputation.git /kaggle/working/well-log-imputation
%cd /kaggle/working/well-log-imputation


In [ ]:
%env CUBLAS_WORKSPACE_CONFIG=:4096:8
%pip install -q -r requirements.txt


## 2. Đường dẫn và thiết lập chung

Chỉ lưu `search_trials.csv`, `best_model_settings.json`, `search_protocol.json` trong `/kaggle/working/well-log-tuning/`. Các lượt thử chạy trong bộ nhớ, không lưu checkpoint hay thư mục riêng. Chạy lại ghi đè file cùng tên.


In [ ]:
from argparse import Namespace
from pathlib import Path
import json
import logging
from zipfile import ZIP_DEFLATED, ZipFile
import sys

import numpy as np
import pandas as pd
from IPython.display import FileLink, display
from sklearn.model_selection import ParameterGrid



In [ ]:
PROJECT_ROOT = Path('/kaggle/working/well-log-imputation')
DATA_ROOT = Path('/kaggle/input/datasets/minhduc0912/geolink-dataset')
OUTPUT_BASE = Path('/kaggle/working/well-log-tuning')
DEVICE = 'cuda'
SEEDS = (129, 219, 309)
BATCH_SIZE = 32
NEURAL_EPOCHS = 100
NEURAL_PATIENCE = 10
SCENARIOS = ['Single', 'Block-20', 'Block-100', 'Entire-Log']

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
from src.experiments.runner import run_experiment
from src.experiments.utils import save_json

In [ ]:
OUTPUT_DIR = OUTPUT_BASE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s', force=True)
print('Output:', OUTPUT_DIR)

## 3. Không gian tìm kiếm

Mỗi cấu hình chạy với tất cả seeds. Conv-SAITS dùng multi-scale nên chỉ tìm `multi_scale_kernels`; `kernel_size` dành cho chế độ một kernel và không tham gia tìm kiếm này.


In [ ]:
SEARCH_SPACES = {
    'xgboost': {
        'max_depth': [2, 3, 4, 5, 6],
        'learning_rate': [0.01, 0.03, 0.05, 0.1],
        'n_estimators': list(range(100, 1001, 100)),
    },
    'brits': {
        'hidden_size': [64, 128, 256],
        'learning_rate': [5e-4, 1e-3],
    },
    'saits': {
        'd_model': [64, 128, 256],
        'learning_rate': [5e-4, 1e-3],
    },
    'conv_saits': {
        'd_model': [64, 128, 256],
        'learning_rate': [5e-4, 1e-3],
        'encoder_channels': [4, 8],
        'multi_scale_kernels': [(3, 5, 7), (3, 7, 15)],
    },
}
FIXED_SETTINGS = {
    'locf': {'batch_size': BATCH_SIZE},
    'xgboost': {'batch_size': BATCH_SIZE},
    'brits': {
        'batch_size': BATCH_SIZE, 'consistency_weight': 0.1, 'weight_decay': 1e-5,
        'mit_weight': 1.0, 'mit_reduction': 'segment', 'masking_strategy': 'mixed',
        'gradient_clip': 1.0, 'validation_metric': 'rmse', 'min_delta': 0.0,
    },
    'saits': {
        'batch_size': BATCH_SIZE, 'd_inner': 128, 'n_layers': 2, 'n_heads': 4,
        'ort_weight': 1.0, 'mit_weight': 1.0, 'mit_reduction': 'segment', 'min_delta': 0.0,
    },
    'conv_saits': {
        'batch_size': BATCH_SIZE, 'd_inner': 128, 'n_layers': 2, 'n_heads': 4,
        'conv_expansion': 2, 'multi_scale_conv': True, 'depth_encoding': False,
        'masking_strategy': 'mixed', 'mit_reduction': 'segment',
        'ort_weight': 1.0, 'mit_weight': 1.0, 'min_delta': 0.0,
        'masking_rate': 0.2, 'independent_shuffle': True,
        'gap_aware_gate': False, 'decoder': 'shared',
    },
}
GRIDS = {name: list(ParameterGrid(space)) for name, space in SEARCH_SPACES.items()}
search_size = pd.DataFrame({
    'configurations': {name: len(grid) for name, grid in GRIDS.items()},
})
search_size['fits'] = search_size['configurations'] * len(SEEDS)
display(search_size)
save_json(OUTPUT_DIR / 'search_protocol.json', {
    'search_spaces': SEARCH_SPACES, 'fixed_settings': FIXED_SETTINGS,
    'seeds': SEEDS, 'epochs': NEURAL_EPOCHS, 'patience': NEURAL_PATIENCE,
    'data_root': str(DATA_ROOT), 'device': DEVICE, 'batch_size': BATCH_SIZE,
    'evaluate_test': False,
    'selection': 'Mean validation RMSE across scenarios and seeds',
})


## 4. Finetune mô hình


In [ ]:
TRAINING_ARGS = dict(
    data_root=str(DATA_ROOT), seeds=SEEDS, device=DEVICE, evaluate_test=False,
    brits_epochs=NEURAL_EPOCHS, brits_patience=NEURAL_PATIENCE,
    saits_epochs=NEURAL_EPOCHS, saits_patience=NEURAL_PATIENCE,
    conv_saits_epochs=NEURAL_EPOCHS, conv_saits_patience=NEURAL_PATIENCE,
)


def validation_score(metrics, name):
    scores = metrics.query(
        "model == @name and split == 'val' and level == 'overall' and units == 'normalized'"
    ).pivot(index='seed', columns='scenario', values='rmse')
    scores = scores.loc[list(SEEDS), SCENARIOS]
    assert np.isfinite(scores.to_numpy()).all()
    per_seed = scores.mean(axis=1)
    return per_seed.mean(), per_seed.std(ddof=1)


## 5. Tìm cấu hình tốt nhất trên validation

In [ ]:
trial_rows = []
best_settings = {'locf': FIXED_SETTINGS['locf'].copy()}

for name, grid in GRIDS.items():
    for trial, parameters in enumerate(grid, 1):
        settings = {name: {**FIXED_SETTINGS[name], **parameters}}
        args = Namespace(
            **TRAINING_ARGS, models=(name,), output_dir=str(OUTPUT_DIR),
            xgb_estimators=parameters.get('n_estimators', 300),
        )
        print(f'{name}: {trial}/{len(grid)} | {parameters}', flush=True)
        trial_metrics = run_experiment(args, PROJECT_ROOT, settings, save_outputs=False)
        score, score_std = validation_score(trial_metrics, name)
        trial_rows.append({
            'model': name, 'trial': trial, 'validation_rmse_mean': score,
            'validation_rmse_std': score_std,
            'parameters': json.dumps(parameters),
        })
        pd.DataFrame(trial_rows).to_csv(OUTPUT_DIR / 'search_trials.csv', index=False)

    model_trials = pd.DataFrame(trial_rows).query('model == @name')
    best_trial = model_trials.sort_values('validation_rmse_mean', kind='stable').iloc[0]
    best_settings[name] = {**FIXED_SETTINGS[name], **grid[int(best_trial['trial']) - 1]}
    save_json(OUTPUT_DIR / 'best_model_settings.json', best_settings)
    print(f"{name}: best validation RMSE = {best_trial['validation_rmse_mean']:.6f}")

trials = pd.DataFrame(trial_rows)
best_trials = trials.sort_values('validation_rmse_mean', kind='stable').drop_duplicates('model')
display(best_trials)
display(pd.DataFrame(best_settings).T)


## 6. Tải kết quả

Xuất bảng so sánh các lượt thử, cấu hình tốt nhất và protocol trong một file ZIP. Điểm chọn cấu hình là validation RMSE trung bình qua scenarios và seeds.


In [ ]:
result_files = ['search_trials.csv', 'best_model_settings.json', 'search_protocol.json']
archive_path = OUTPUT_DIR.parent / 'well-log-tuning.zip'
with ZipFile(archive_path, 'w', compression=ZIP_DEFLATED) as archive:
    for name in result_files:
        archive.write(OUTPUT_DIR / name, arcname=name)
display(FileLink(str(archive_path)))
